> `oeai_mod_police.ipynb`
> [20251105.1]
> *Police notebook*

In [ ]:
# To map Postcodes to lng and lat
%pip install pgeocode

In [ ]:
%run ./oeai_mod_police_env_var

In [ ]:
# Initialise Logging
oeai.notebook = SimpleNamespace(
    name = "oeai_mod_police.ipynb",
    buildversion = "20251105.1",
    buildtimestamp = "2025-11-05T16:00:00Z",
)
oeai.log.init()
oeai.log.start_block("Notebook Init")

# Functions

In [ ]:
import requests, json, time, pgeocode
from pyspark.sql import Row, functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType, StringType

In [ ]:
def get_lat_lon(postcode):
    if postcode:
        location = nomi.query_postal_code(postcode)
        if location is not None and not location.empty:
            return Row(lat=float(location.latitude or 0), lng=float(location.longitude or 0))
    return Row(lat=None, lng=None)

In [ ]:
# Define the API calling function with rate-limiting
def fetch_data(lat, lng, key, endpoint):
    
    # Check if latitude or longitude are None (null)
    if lat is None or lng is None:
        return [(key, None)]
    
    url = f"https://data.police.uk/api/{endpoint}?lat={lat}&lng={lng}"
    
    # Print the URL to the screen
    oeai.log.debug(
        f"Calling API with URL: {url}", 
        url=url, 
        lat=lat,
        lng=lng,
        endpoint=endpoint,
        function= "fetch_data"
    )
    
    # Make the API call
    response = requests.get(url)
    
    if response.status_code == 200:
        # Parse the response JSON and add key
        rows = response.json()
        # If crimes are found, attach the key to each crime
        return [(key, row) for row in rows]
    else:
        return [(key, None)]  # Return None if the API call fails

In [ ]:
# Apply the function to the DataFrame with an optional limit
def process_with_rate_limit(df, endpoint, limit=None):
    results = []
    counter = 0  # Initialize the counter
    
    # Collect all rows, but limit the number of rows if 'limit' is provided
    rows = df.collect()[:limit] if limit else df.collect()
    
    # Iterate over each row and make the API call
    for row in rows:
        # Call the API and get the results
        result = fetch_data(
            lat=row.lat, 
            lng=row.lng, 
            key=row.coord, 
            endpoint=endpoint
        )
        results.extend(result)
        
        # Increment and print the counter for each row processed
        counter += 1
        print(f"Processed {counter} rows.")
        
        # Sleep to respect the rate limit: 15 requests per second (1/15 sec delay)
        time.sleep(1 / 15)
    
    # Create a DataFrame from the results
    rows_out = [Row(coord=x[0], response=json.dumps(x[1])) for x in results]
    df_out = spark.createDataFrame(rows_out)
    
    return df_out

In [ ]:
oeai.log.end_block()
oeai.log.checkpoint("Notebook Init")

# Processing

In [ ]:
oeai.log.start_block("Students → Postcode")

# Load StudentExtended Table and gather distinct postcodes

# Load dim_StudentExtended
df_dim_StudentExtended = spark.read.format("delta").load(os.path.join(silver_path, "dim_StudentExtended"))

# Distinct postcodes
df_postcode = (df_dim_StudentExtended
    .filter(F.col("Postcode").isNotNull() & (F.length(F.trim(F.col("Postcode"))) > 0))
    .select("Postcode").distinct()
)

oeai.log.end_block()
oeai.log.checkpoint("Students → Postcode")

In [ ]:
oeai.log.start_block("Postcode → (lat, long)")

# Initialize pgeocode for UK postcodes
nomi = pgeocode.Nominatim("GB")

# Define schema for UDF return type
geo_schema = StructType([
    StructField("lat", DoubleType(), True),
    StructField("lng", DoubleType(), True)
])

# Convert function to UDF
get_lat_lon_udf = F.udf(get_lat_lon, geo_schema)

# Apply the UDF to get coordinates
df_postcode_coord = (
    df_postcode
    .withColumn("coord", get_lat_lon_udf(col("Postcode")))
)

# Select distinct coordinates and split into latitude/longitude
df_coord = (
    df_postcode_coord
    .select("coord").distinct()
    .withColumn("lat", col("coord.lat"))
    .withColumn("lng", col("coord.lng"))
    .filter(~(F.col("lat").isNull() | F.col("lng").isNull() | F.isnan(F.col("lat")) | F.isnan(F.col("lng"))))
)

oeai.log.end_block()
oeai.log.checkpoint("Postcode → (lat, long)")

In [ ]:
oeai.log.start_block("Fetch data")

oeai.log.start_block("Fetch CRIME data")
df_crimes   = process_with_rate_limit(df_coord, endpoint="crimes-street/all-crime")
oeai.log.end_block()
oeai.log.checkpoint("Fetch CRIME data")

oeai.log.start_block("Fetch STOPS data")
df_stops    = process_with_rate_limit(df_coord, endpoint="stops-street")
oeai.log.end_block()
oeai.log.checkpoint("Fetch STOPS data")

oeai.log.end_block()
oeai.log.checkpoint("Fetch data")

In [ ]:
oeai.log.start_block("Pivot Crime stats")

# Define Schema - Crime
street_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
])

location_schema = StructType([
    StructField("latitude", StringType(), True),
    StructField("longitude", StringType(), True),
    StructField("street", street_schema, True),
])

crime_schema = StructType([
    StructField("category", StringType(), True),
    StructField("location_type", StringType(), True),
    StructField("location", location_schema, True),
    StructField("context", StringType(), True),
    StructField("outcome_status", StringType(), True),
    StructField("persistent_id", StringType(), True),
    StructField("id", IntegerType(), True),
    StructField("location_subtype", StringType(), True),
    StructField("month", StringType(), True),
])

# Build Pivot
df_crimes_pivot = (
    df_crimes
    .withColumn("crime_json", F.from_json(df_crimes["response"], crime_schema))
    .select(
       "coord",
        F.col("crime_json.category").alias("category"),
        F.col("crime_json.month").alias("month")
    )
    .filter(
        (F.col("category").isNotNull()) |
        (F.col("month").isNotNull())
    )
    .groupBy("coord", "month", "category").count()
    .groupBy("coord", "month").pivot("category").agg(F.sum("count"))
)

# Rename Columns
for col_name in df_crimes_pivot.columns:
    if col_name not in ["coord", "month"]:  # Skip the key and month columns
        df_crimes_pivot = df_crimes_pivot.withColumnRenamed(col_name, f"Crime - {col_name}")

# display(df_crimes_pivot)

oeai.log.end_block()
oeai.log.checkpoint("Pivot Crime stats")

In [ ]:
oeai.log.start_block("Pivot Stop & Search stats")

# Define Schema - Stops
stop_data_schema = StructType([
    StructField("object_of_search", StringType(), True)
])

# Build Pivot
df_stops_pivot = (
    df_stops
    .withColumn("stop_json", F.from_json(F.col("response"), stop_data_schema))
    .select(
        "coord",
        F.col("stop_json.object_of_search").alias("object_of_search")
    )
    .filter(F.col("object_of_search").isNotNull())
    .groupBy("coord", "object_of_search").count()
    .groupBy("coord").pivot("object_of_search").agg(F.sum("count"))
)

# Rename Columns
for col_name in df_stops_pivot.columns:
    if col_name != "coord":  # Skip the key column
        df_stops_pivot = df_stops_pivot.withColumnRenamed(col_name, f"Stop - {col_name}")

# display(df_stops_pivot)

oeai.log.end_block()
oeai.log.checkpoint("Pivot Stop & Search stats")

## Final

In [ ]:
oeai.log.start_block("Combine pivots")

# Join df_pivoted_stops onto df_pivoted_crimes based on key to have one table with 1 row per postcode
df_final = (
    df_postcode_coord
    .join(df_crimes_pivot, "coord", "left")
    .join( df_stops_pivot, "coord", "left")
)

# display(df_final)

oeai.log.end_block()
oeai.log.checkpoint("Combine pivots")

In [ ]:
oeai.log.start_block("Write Output")

# Write the result to a new parquet file
df_final.write.mode("overwrite").format("parquet").save(os.path.join(gold_path, "fact_Crime"))

oeai.log.end_block()
oeai.log.checkpoint("Write Output")

In [ ]:
oeai.log.end_block(index=0, include_target=True)
oeai.log.shutdown()
oeai.log.checkpoint()